In [414]:
import torch
import torch.nn as nn
import numpy as np

In [415]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cpu'

In [416]:
data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [1, 0, 1, 0, 1, 0] # 1: 긍정, 0: 부정
}

In [417]:
import pandas as pd
df = pd.DataFrame(data)
df.head(1)

,reviews,ratings
0,이 영화 정말 좋아 최고야,1


In [418]:
# 전처리
import re
def clean_data(text):
    return re.sub(r'[^가-힣\s]', '', text)
df['clean'] = df['reviews'].apply(lambda x : clean_data(x))

# 토큰분류 형태소
from konlpy.tag import Okt
okt = Okt()
def kor_tokened(text):    
    return [word for word, pos in okt.pos(text,stem=True) if len(word) > 1 and pos in ['Noun','Verb','Adjective']]

df['tokens'] = df['clean'].apply(lambda x : kor_tokened(x))

In [419]:
df['tokens']

0        [영화, 정말, 좋다, 최고]
1      [시간, 아깝다, 쓰레기, 영화]
2          [배우, 연기, 훌륭하다]
3        [스토리, 지루하다, 뻔하다]
4    [인생, 최고, 명작, 이다, 추천]
5       [주다, 보기, 아깝다, 졸작]
Name: tokens, dtype: object

In [420]:
# 단어인덱싱
vocab = {
    '<PAD>' : 0,
    '<UNK>' : 1
}
def make_vocab(tokens):
    for token in tokens:    
        if token not in vocab:
            vocab[token] = len(vocab)
df['tokens'].apply(lambda x : make_vocab(x))

0    None
1    None
2    None
3    None
4    None
5    None
Name: tokens, dtype: object

In [421]:
# padding  길이 맞추기 
MAX_LEN = 5
def to_sequence(tokens):
    # 토큰화 - 패딩    
    x = [vocab.get(word,1) for word in tokens ]
    if len(x) < MAX_LEN:
        x = x + [0]*(MAX_LEN-len(x))
    else:
        x = x[:MAX_LEN]
    return x

df['pad_tokens'] =  df['tokens'].apply(lambda x : to_sequence(x))
df

,reviews,ratings,clean,tokens,pad_tokens
0,이 영화 정말 좋아 최고야,1,이 영화 정말 좋아 최고야,"[영화, 정말, 좋다, 최고]","[2, 3, 4, 5, 0]"
1,시간 아까운 쓰레기 영화,0,시간 아까운 쓰레기 영화,"[시간, 아깝다, 쓰레기, 영화]","[6, 7, 8, 2, 0]"
2,배우들 연기가 너무 훌륭해요,1,배우들 연기가 너무 훌륭해요,"[배우, 연기, 훌륭하다]","[9, 10, 11, 0, 0]"
3,스토리가 지루하고 뻔하다,0,스토리가 지루하고 뻔하다,"[스토리, 지루하다, 뻔하다]","[12, 13, 14, 0, 0]"
4,인생 최고의 명작입니다 추천,1,인생 최고의 명작입니다 추천,"[인생, 최고, 명작, 이다, 추천]","[15, 5, 16, 17, 18]"
5,돈 주고 보기 아까운 졸작,0,돈 주고 보기 아까운 졸작,"[주다, 보기, 아깝다, 졸작]","[19, 20, 7, 21, 0]"


In [422]:
# 데이터셋, -- > Tensor타입변경 
# 데이터로더 --> 배치단위로 학습
from torch.utils.data  import Dataset, DataLoader
class ReviewMovieDataset(Dataset):
    def __init__(self,x,y):
        self.x = torch.LongTensor(x)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, index):
        return self.x[index], self.y[index]
x = df['pad_tokens'].tolist();  y = df['ratings'].tolist()
dataset = ReviewMovieDataset(x,y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)        

In [423]:
print( next(iter(dataloader)) )

[tensor([[ 2,  3,  4,  5,  0],
        [19, 20,  7, 21,  0]]), tensor([1., 0.])]


In [424]:
VOCAB_SIZE = len(vocab)

In [425]:
import torch.nn.functional as F
class TextCNN(nn.Module):
    def __init__(self, vocab_size,embedding_dim, filter_sizes, num_fillers):
        super().__init__()        
        self.embedding = nn.Embedding(num_embeddings = vocab_size, embedding_dim=embedding_dim)
        # 합성곱 계층  텍스트는 이미지의 흑백처럼 채널을 1로 해서 사용
        # in_channel = 1(흑백이미지처럼 채널이 1)
        # kernel_size =(n-gram크기, 임베딩차원) -> 단어가 쪼개지지 않도록 width embedding dim과 동일
        self.convs =  nn.ModuleList(
                        [nn.Conv2d(in_channels=1, out_channels = num_fillers,
                                kernel_size=(fs, embedding_dim))
                                for fs in filter_sizes]
        )
        # 완전연결층,FC,분류기 입력
        # 추출된특징들을 모두 이어 붙임
        self.fc = nn.Linear(len(filter_sizes)*num_fillers, 1) # 이진분류
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        # x.shape (batch, seq_len)  (2, 5)
        # 임베딩 적용 -> (batch,seq_len,embedding_dim) (2,5,embedding_dim)
        x = self.embedding(x)
        # conv2d 입력(batch, channel, height, width) 채널정보를 받아야해서
        # shape : (batch,1,seq_len, embedding_dim)
        x = x.unsqueeze(1)
        pooled_outputs = []
        for conv in self.convs:
            # 합성곱, Relu 연산  shape : (batch,num_filters, seq_len-filter_size+1 ,1)
            c = F.relu(conv(x))  # shape(batch, num_filters,seq_len-filter_size+1)
            c = c.squeeze(3)
            # 최대폴링 : 문장에서 가장 강력한 특징 1개만 추출
            p = F.max_pool1d(c, c.size(2)) # shape (batch, num_filters,1)
            pooled_outputs.append(p.squeeze(2)) # shape (batch,num_filters)
        # 분류(추출된 피처들을 concatenate 후 FC레이어 통과)
        x_cat = torch.concatenate(pooled_outputs, dim=1) # shape(Batch, num_filters*len(filter_sizes))
        x_cat = self.dropout(x_cat)

        logits = self.fc(x_cat) # shape (Batch,1)
        return logits.squeeze(1) # shape (batch)

In [426]:
# 모델 셋업
VOCAB_SIZE = len(vocab)
EMBED_DIM = 16 # 단어를 16차원 벡터로 표현
FILTER_SIZES = [2,3,4,5]  # 바이어그램(2), 트라이그램(3)사용
NUM_FILTERS = 10  # 각 사이즈별 필터 개수
model = TextCNN(VOCAB_SIZE,EMBED_DIM,FILTER_SIZES,NUM_FILTERS)

In [427]:
model

TextCNN(
  (embedding): Embedding(22, 16)
  (convs): ModuleList(
    (0): Conv2d(1, 10, kernel_size=(2, 16), stride=(1, 1))
    (1): Conv2d(1, 10, kernel_size=(3, 16), stride=(1, 1))
    (2): Conv2d(1, 10, kernel_size=(4, 16), stride=(1, 1))
    (3): Conv2d(1, 10, kernel_size=(5, 16), stride=(1, 1))
  )
  (fc): Linear(in_features=40, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [428]:
from torch.optim import Adam
criterion = nn.BCEWithLogitsLoss()  # 내부에 sigmoid 포함
optimizer = Adam(model.parameters(), lr=1e-3)

In [429]:
from tqdm import tqdm
epochs = 30
model.train()
for epoch in tqdm(range(epochs)):
    total_loss = 0.0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions,batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'epoch : {epoch+1} / {epochs} loss : {( total_loss / len(dataloader) ):.4f}')

  0%|          | 0/30 [00:00<?, ?it/s]

epoch : 1 / 30 loss : 0.7209
epoch : 2 / 30 loss : 0.7423
epoch : 3 / 30 loss : 0.7259
epoch : 4 / 30 loss : 0.6358
epoch : 5 / 30 loss : 0.6663


 20%|██        | 6/30 [00:00<00:00, 54.75it/s]

epoch : 6 / 30 loss : 0.5804
epoch : 7 / 30 loss : 0.6085
epoch : 8 / 30 loss : 0.6921
epoch : 9 / 30 loss : 0.5562
epoch : 10 / 30 loss : 0.5858
epoch : 11 / 30 loss : 0.5335


 43%|████▎     | 13/30 [00:00<00:00, 59.02it/s]

epoch : 12 / 30 loss : 0.4761
epoch : 13 / 30 loss : 0.3981
epoch : 14 / 30 loss : 0.4835
epoch : 15 / 30 loss : 0.4993
epoch : 16 / 30 loss : 0.3624
epoch : 17 / 30 loss : 0.4249
epoch : 18 / 30 loss : 0.4423


 67%|██████▋   | 20/30 [00:00<00:00, 60.59it/s]

epoch : 19 / 30 loss : 0.4506
epoch : 20 / 30 loss : 0.5021
epoch : 21 / 30 loss : 0.4692
epoch : 22 / 30 loss : 0.3347
epoch : 23 / 30 loss : 0.2934
epoch : 24 / 30 loss : 0.3454


100%|██████████| 30/30 [00:00<00:00, 60.24it/s]

epoch : 25 / 30 loss : 0.3950
epoch : 26 / 30 loss : 0.4019
epoch : 27 / 30 loss : 0.2377
epoch : 28 / 30 loss : 0.2030
epoch : 29 / 30 loss : 0.2022
epoch : 30 / 30 loss : 0.3081


In [430]:
# 예측
model.eval()
test_reivew = '돈 주고 보기 아까운 졸작'#'이 영화 정말 좋아 최고야'
test_reivew = [
    vocab.get(word,1)
        for word in [word for word, pos in okt.pos(test_reivew,stem=True) if len(word) > 1]
]
if len(test_reivew) < MAX_LEN:
    test_reivew = test_reivew + [0]*(MAX_LEN - len(test_reivew))
test_reivew = torch.LongTensor(test_reivew)
test_reivew = test_reivew.unsqueeze(0)
with torch.no_grad():
    logits = model(test_reivew)
    prob = torch.sigmoid(logits)[0].item()
    print( '긍정' if prob > 0.5 else '부정' , prob)  

    

부정 0.2006908506155014


### 영화리뷰데이터 적용

In [431]:
url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
df = pd.read_csv(url)
df.head()

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워
4,재미있다,10,2018.10.20,인피니티 워


In [432]:
# rating 1~3  0   10~8  1  클래스 불균형 없게 40 : 40  --> 60
zero_mask = (1<=df['rating']) & (df['rating'] <=3)
zero_df_20 = df[zero_mask][:40]
zero_df_20['rating'] = 0


one_mask = (8<=df['rating']) & (df['rating'] <=10)
one_df_20 = df[one_mask][:40]
one_df_20['rating'] = 1

In [433]:
new_df = pd.concat((zero_df_20,one_df_20))
new_df

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,0,2018.10.29,인피니티 워
41,이건 뭐~ 이 정도 돈과 출연진 가지고 이렇게 망칠 수도 있구나를 보여주는 대표작...,0,2018.08.24,인피니티 워
49,중간중간 넘 지루해~~~~~~~~~~~,0,2018.08.16,인피니티 워
51,비젼보다 더 강한 스파이 ㅋㅋㅋㅋ,0,2018.08.13,인피니티 워
72,다잡은 보스를 쪼렙 한놈이 어이없는 열폭실수로 못죽여서 우주 생명체 50%가 죽는다...,0,2018.08.06,인피니티 워
...,...,...,...,...
53,"물론 타노스가 생명체의 절반을 날리는 데 성공한 것은 많이 아쉽지만, 그래도 정말 ...",1,2018.08.12,인피니티 워
54,결말이 다음 편을 위한 떡밥이라 만점주기는 머한... 그래도 이렇게 화려한 캐스팅을...,1,2018.08.12,인피니티 워
58,타노스는 이미 할일을 다 했다. 남은건 어벤져스가 과거로 돌아가 어느 부분에서 타...,1,2018.08.11,인피니티 워
60,당연히 10점 ㄱㄱ,1,2018.08.11,인피니티 워


In [434]:
# 1. 학습 테스트 데이터 분리
# 2. 데이터전처리(불용어,특수문자, 구두점 등등)  정규표현식
# 3. 토큰화 - 단어분리(okt 형태소별)
# 4. 단어사전 - 단어인덱스
# 5. 패딩    - 문서의 길이를 맞춤
# 6. 텐서변환 - 학습용 데이터 타입일치(torch tensor사용)
# 7. 데이터셋 + 데이터로더(학습용 테스용)
# 8. 모델셋팅(nn.Module상속받아서 클래스구성)  cnn, n-gram(필터-도장)  (n-gram, 스퀀스길이)
# 9. 하이퍼 파라메터 설정 - 옵티마이져, 러닝레이트, 손실함수 등등
# 10. 학습 - 데이터로더를 이용한 학습
# 11. 평가 - 데이터로드를 이용한 평가
# 12. 추론

In [435]:
# 1. 학습 테스트 데이터 분리
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(new_df['review'], new_df['rating'],test_size=0.2, random_state=42)

# 1. 데이터전처리
import re
def clean_data(text):
    return re.sub(r'[^가-힣\s]', '', text).strip()

# 2. 토큰화
from konlpy.tag import Okt
okt = Okt()
def korea_tokenizer(text):
    return [
        word
    for word, pos in okt.pos(text,stem=True)
    if len(word) > 1 and pos in ['Noun','Verb','Adjective']
    ]
# 단어사전
vocab = {'<PAD>':0, '<UNK>' : 1}
def make_vocab(tokens):
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab)


clean_data_tokens_list = []
for text in new_df['review'].tolist():
    clean_txt = clean_data(text)
    tokens = korea_tokenizer(clean_txt)
    if len(tokens) > 0:
        clean_data_tokens_list.append(tokens)


# clean_data_tokens_list = 
# [
#         korea_tokenizer(c_data) for c_data in 
#             [clean_data(text) for text in new_df['review'].tolist()] 
#     ]

for c_lists in clean_data_tokens_list: #[ ['좋아','영화'] ...  ]  
    if len(c_lists) > 0:
        make_vocab(c_lists)
# ---------  단어사전 완성 ------------------------

def review_pad_sequence(review,sequence_len):
    '''한글문장->전처리->한글토큰->단어사전을 이용한 인덱싱'''
    review = clean_data(review)
    review = [vocab.get(token,1) for token in korea_tokenizer(review)]        
    if len(review) < sequence_len:
        return review + [0]*(sequence_len - len(review))
    else:
        return review[:sequence_len]



# # 4. 텐서변환
# # 5. 데이터로더
from torch.utils.data import DataLoader, Dataset
class TextDataSet(Dataset):
    '''
        review_lists:List

        targets:List
    '''    
    def __init__(self,review_lists,targets,sequence_len=10):
        self.x = torch.LongTensor([ review_pad_sequence(review,sequence_len) for review in review_lists])
        self.y = torch.FloatTensor(targets)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, index):
        return self.x[index], self.y[index]  
SEQ_LEN = 10    
x_train_dataset = TextDataSet(x_train.tolist(),y_train.tolist(),SEQ_LEN)
x_train_dataloader = DataLoader(x_train_dataset, batch_size=5, shuffle=True)

x_test_dataset = TextDataSet(x_test.tolist(),y_test.tolist(),SEQ_LEN)
x_test_dataloader = DataLoader(x_test_dataset, batch_size=5)

temp_x, temp_y = next(iter(x_train_dataloader))
temp_x.shape, temp_y.shape

(torch.Size([5, 10]), torch.Size([5]))

In [436]:
[ review_pad_sequence(review,10) for review in x_train.tolist()]

[[63, 369, 370, 371, 193, 7, 140, 3, 372, 134],
 [342, 342, 73, 343, 344, 345, 0, 0, 0, 0],
 [240, 309, 0, 0, 0, 0, 0, 0, 0, 0],
 [257, 20, 105, 258, 259, 20, 260, 261, 262, 263],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [349, 91, 350, 131, 351, 0, 0, 0, 0, 0],
 [56, 57, 58, 59, 60, 3, 61, 62, 63, 64],
 [286, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [230, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [156, 192, 305, 12, 60, 53, 0, 0, 0, 0],
 [7, 20, 20, 12, 0, 0, 0, 0, 0, 0],
 [307, 308, 12, 0, 0, 0, 0, 0, 0, 0],
 [163, 164, 20, 20, 69, 165, 73, 166, 167, 164],
 [3, 107, 36, 105, 0, 0, 0, 0, 0, 0],
 [310, 32, 40, 105, 0, 0, 0, 0, 0, 0],
 [333, 73, 411, 20, 12, 60, 279, 277, 319, 0],
 [74, 75, 76, 77, 78, 79, 80, 81, 51, 82],
 [63, 40, 39, 32, 116, 300, 0, 0, 0, 0],
 [277, 305, 306, 20, 0, 0, 0, 0, 0, 0],
 [168, 169, 0, 0, 0, 0, 0, 0, 0, 0],
 [352, 187, 0, 0, 0, 0, 0, 0, 0, 0],
 [181, 182, 183, 184, 185, 186, 187, 20, 188, 183],
 [279, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [131, 140, 0, 0, 0, 0, 0, 0, 0, 0],
 [285, 277, 397, 398, 

In [437]:
# 모델은 상단에 정의된 클래스 이용
# 모델 셋업
VOCAB_SIZE = len(vocab)
EMBED_DIM = 16 # 단어를 16차원 벡터로 표현
FILTER_SIZES = [2,3,4,5]  # 바이어그램(2), 트라이그램(3)사용
NUM_FILTERS = 10  # 각 사이즈별 필터 개수
model = TextCNN(VOCAB_SIZE,EMBED_DIM,FILTER_SIZES,NUM_FILTERS)
optimizer = Adam(model.parameters(), lr=0.01)


from tqdm import tqdm
epochs = 47
model.train()
for epoch in tqdm(range(epochs)):
    total_loss = 0.0
    for batch_x, batch_y in x_train_dataloader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions,batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'epoch : {epoch+1} / {epochs} loss : {( total_loss / len(x_train_dataloader) ):.4f}')

  6%|▋         | 3/47 [00:00<00:03, 11.16it/s]

epoch : 1 / 47 loss : 0.8030
epoch : 2 / 47 loss : 0.6802
epoch : 3 / 47 loss : 0.5335


 11%|█         | 5/47 [00:00<00:03, 11.67it/s]

epoch : 4 / 47 loss : 0.4112
epoch : 5 / 47 loss : 0.3790
epoch : 6 / 47 loss : 0.2890


 19%|█▉        | 9/47 [00:00<00:03, 12.65it/s]

...

 91%|█████████▏| 43/47 [00:03<00:00, 13.72it/s]

epoch : 42 / 47 loss : 0.0220
epoch : 43 / 47 loss : 0.0287
epoch : 44 / 47 loss : 0.0925


 96%|█████████▌| 45/47 [00:03<00:00, 11.29it/s]

epoch : 45 / 47 loss : 0.0660
epoch : 46 / 47 loss : 0.0717
epoch : 47 / 47 loss : 0.0273


100%|██████████| 47/47 [00:03<00:00, 12.25it/s]


In [438]:
# 평가
model.eval()
with torch.no_grad():
    total_loss = 0.0
    for batch_x, batch_y in x_test_dataloader:        
        print(batch_x,batch_y)
        predictions = model(batch_x)
        loss = criterion(predictions,batch_y)        
        total_loss += loss.item()
    print(f'{ (total_loss / len(x_test_dataloader)):.4f}')

tensor([[220, 105, 221,   3, 222, 223, 120, 224,   3,   0],
        [  2,   3,   4,   5,   0,   0,   0,   0,   0,   0],
        [ 16,  35, 172,  60,  37, 173, 174, 175, 105, 176],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [ 36,   0,   0,   0,   0,   0,   0,   0,   0,   0]]) tensor([0., 0., 0., 0., 0.])
tensor([[207, 208,   3, 209, 210, 211, 212, 105, 213, 134],
        [110,  36, 111, 112, 113, 114, 115, 116, 112, 117],
        [313,  53, 361, 257,  20, 105, 282,   3,  69, 362],
        [ 44,  45,  46,  47,  48,  49,  50,  51,  52,  53],
        [ 37, 139,   0,   0,   0,   0,   0,   0,   0,   0]]) tensor([0., 0., 1., 0., 0.])
tensor([[299,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [226, 107,   3,  37, 227, 228, 229,   0,   0,   0],
        [201,   3, 353, 354, 259,  20, 355, 117, 155, 312],
        [231, 141, 232,   0,   0,   0,   0,   0,   0,   0],
        [ 25, 189, 357, 279,  12,  73, 358,   0,   0,   0]]) tensor([1., 0., 1., 0., 1.])
tensor([[2

In [439]:
# 추론
test_x = '중간중간 넘 지루해'
test_x = review_pad_sequence(test_x,10)
test_x = torch.LongTensor(test_x)
test_x = test_x.unsqueeze(0)
logit = model(test_x)
prob = torch.sigmoid(logit).item()
prob

4.5687264105254144e-07